# Spatial-bin event propagation

Approximately 4,600 cells are sampled across 36 coordinate clusters. A lag plane is fitted separately to every population event, preventing oppositely directed events from cancelling in an event-triggered mean.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from yumeow_plot import get_fig
ROOT = Path('..').resolve()
RUNS = ROOT / 'logs/scripts/spatial_bin_event_propagation'
OUT = ROOT / 'figures/261027空间事件传播'
OUT.mkdir(parents=True, exist_ok=True)
archive = np.load(RUNS / 'initial/event_arrays.npz')
centers = archive['centers']
run_names = ['initial', 'seed_20261010', 'seed_20261011', 'bins_25', 'bins_49']
summaries = []
for run in run_names:
    table = pd.read_csv(RUNS / run / 'summary.csv')
    table['run'] = run
    summaries.append(table)
summaries = pd.concat(summaries, ignore_index=True)

In [2]:
mean_lags = {state: archive[f'{state}_event_lags_seconds'].mean(axis=0) for state in ['Awake', 'NREM']}
limit = max(np.max(np.abs(value)) for value in mean_lags.values())
colors = {'Awake': '#cc5544', 'NREM': '#4477aa'}
fig, axes = get_fig(RN=1, CN=3, AW=4, AH=4, LM=5, RM=5, TM=5, BM=5, VS=5, HS=5, fontsize=14, dpi=300, lw=1)
ax_loader = iter(axes)
for state in ['Awake', 'NREM']:
    ax = next(ax_loader)
    scatter = ax.scatter(centers[:, 0], centers[:, 1], c=mean_lags[state], cmap='RdBu_r', vmin=-limit, vmax=limit, s=85, edgecolor='0.2', lw=.4)
    ax.set(xlabel='Projected x', ylabel='Projected y', title=f'{state} mean event lag')
    ax.set_aspect('equal')
fig.colorbar(scatter, ax=list(axes[:2]), label='Mean peak lag (s)', shrink=.7)
ax = next(ax_loader)
for state, offset in [('Awake', -.07), ('NREM', .07)]:
    part = summaries[summaries.state == state]
    x = np.arange(len(run_names)) + offset
    ax.scatter(x, part.set_index('run').loc[run_names].median_event_plane_r2, s=55, color=colors[state], label=state)
ax.set_xticks(range(len(run_names)), ['seed 0', 'seed 1', 'seed 2', '25 bins', '49 bins'], rotation=40, ha='right')
ax.set(ylabel='Median event lag-plane $R^2$', title='Sampling robustness')
ax.legend(frameon=False)
fig.savefig(OUT / 'spatial_event_propagation.png')
fig.savefig(OUT / 'spatial_event_propagation.svg')
fig.savefig(OUT / 'spatial_event_propagation.pdf')
plt.show()

Across three independent cell samples and 25/36/49 spatial bins, median per-event lag-plane R² ranges from 0.141–0.180 in Awake and 0.137–0.190 in NREM; every spatial-permutation test reaches p≈0.002. Event directions are weakly but significantly concentrated and their circular means remain within 112°–128° in Awake and 114°–132° in NREM.

Raw depth alone explains only median R² 0.023–0.052, and the unknown fourth raw coordinate only 0.013–0.034, much less than the projected spatial plane. Thus the effect is not dominated by those candidate scan coordinates. However, the similar preferred direction across states leaves a fixed acquisition-order artifact possible until scan geometry is known or an independent recording validates the direction.